# ARC-v0.35 — NQ GTE-base 768-d Frozen Mechanism Replication

This prospective replication tests the strongest remaining external-validity concern in the current SIGIR manuscript: all prior encoder evidence uses compact 384-d models.

**Frozen design**
- BEIR Natural Questions
- `thenlper/gte-base`; runtime dimension must equal 768 or the study stops
- representation: IVF-PQ64 @ nprobe=64 vs IVF-SQ8 @ nprobe=64
- search effort: IVF-SQ8 @ FIT-calibrated nprobe vs IVF-SQ8 @ nprobe=64
- anchored centroid feedback, H=4
- 8-policy structural subset: alpha ∈ {0.1,0.3,0.5,0.7} × {mean-k20, softmax-k20-tau0.1}
- primary: VALID query-level mean of `H3_abs_rep - H3_abs_search`
- 10,000 paired-query bootstrap

PQ64 preserves the previous subvector width: 384/32 = 768/64 = 12. It is not a cost-matching claim. FIT selects nprobe; VALID remains untouched until calibration is sealed. Positive, null, and reversed outcomes are all retained.


In [1]:
import sys, subprocess
def install(*p): subprocess.check_call([sys.executable,'-m','pip','install','-q',*p])
install('faiss-cpu==1.12.0','sentence-transformers>=5.0,<6','transformers>=4.55,<5','accelerate>=1.5','pyarrow','tqdm','requests')


In [2]:
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import gc, hashlib, json, math, os, random, re, zipfile
import faiss, numpy as np, pandas as pd, requests, torch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
SEED=20260835; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
MODEL_ID='thenlper/gte-base'; DIM=768; NLIST=4096; PQ_M=64; PQ_NBITS=8
HIGH_NPROBE=64; NPROBE_GRID=[1,2,4,8,16,32]; TRAIN_DOCS=250000
TOPK=100; UK=10; H=4; BLOCK=50000; BOOT=10000
POLICIES=[{'alpha':a,'kind':k,'k':20,'tau':(.1 if k=='softmax' else None)} for a in [.1,.3,.5,.7] for k in ['mean','softmax']]
SALT='ARC-v0.35-NQ-GTEBASE-768D-FROZEN-SPLIT-v1'
PRIOR_384={'mean':0.011868,'ci95':[0.009922,0.013794]}
def sha_file(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(16*1024*1024),b''): h.update(b)
 return h.hexdigest()
def split_score(q): return int(hashlib.sha256(f'{SALT}|{q}'.encode()).hexdigest(),16)
def norm_rows(x):
 x=np.asarray(x,dtype=np.float32); return x/np.maximum(np.linalg.norm(x,axis=1,keepdims=True),1e-12)
def ols(y):
 y=np.asarray(y,float); x=np.arange(len(y),dtype=float); xc=x-x.mean(); return float(np.dot(xc,y-y.mean())/np.dot(xc,xc))
def boot_mean(x,seed):
 x=np.asarray(x,float); rng=np.random.default_rng(seed); n=len(x); z=np.empty(BOOT)
 for i in range(BOOT): z[i]=x[rng.integers(0,n,n)].mean()
 return {'n':n,'mean':float(x.mean()),'ci95':[float(v) for v in np.quantile(z,[.025,.975])]}


In [3]:
from google.colab import drive
DRIVE=Path('/content/drive/MyDrive')
if not DRIVE.is_dir(): drive.mount('/content/drive')
ROOT=DRIVE/'rag-pq-checkpoints'/'arc-v0'/'nq-gte-base-768d-frozen-replication-v035'
CACHE=ROOT/'cache'; EMB=CACHE/'corpus_embeddings_f16'; IDX=CACHE/'indexes'; RUNS=ROOT/'runs'
for p in [ROOT,CACHE,EMB,IDX,RUNS]: p.mkdir(parents=True,exist_ok=True)
RUN=RUNS/datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S'); RUN.mkdir(parents=True,exist_ok=False)
protocol={'study_id':'ARC-v0.35','status':'FROZEN_BEFORE_GTE_BASE_EFFECTIVENESS','dataset':'BEIR NQ','encoder':MODEL_ID,'dimension':DIM,'split_rule':'SHA256 salt; first half FIT, second half VALID','salt':SALT,'index':{'nlist':NLIST,'pq_m':PQ_M,'pq_nbits':PQ_NBITS,'high_nprobe':HIGH_NPROBE,'nprobe_grid':NPROBE_GRID,'train_docs':TRAIN_DOCS},'representation':'PQ64@64 vs SQ8@64','search_effort':'SQ8@FIT-selected-nprobe vs SQ8@64','calibration':'FIT-only minimum absolute mismatch in mean one-shot nDCG@10 loss; ties choose smaller nprobe','feedback':{'operator':'anchored centroid','H':H,'policies':POLICIES},'primary':'VALID query-level mean representation-minus-search H3_abs after within-query 8-policy averaging','inference':'10000 paired-query bootstrap','retention':'retain positive/null/reversed; no post-VALID retuning','guardrail':'positive result transfers to one 768-d encoder only; not dimension invariance or cost matching','prior_384':PRIOR_384}
pp=RUN/'V035_FROZEN_PROTOCOL.json'; pp.write_text(json.dumps(protocol,indent=2,sort_keys=True)); PROTOCOL_SHA=sha_file(pp); (RUN/'V035_PROTOCOL_SHA256.txt').write_text(PROTOCOL_SHA)
print('RUN',RUN,'protocol',PROTOCOL_SHA)


Mounted at /content/drive
RUN /content/drive/MyDrive/rag-pq-checkpoints/arc-v0/nq-gte-base-768d-frozen-replication-v035/runs/20260829-130645 protocol 179a2def74f4f038468440b631f1e656f393127e944f23a1b86f5247e61c4143


In [4]:
RAW=Path('/content/arc-v035-nq'); RAW.mkdir(exist_ok=True); NQ=RAW/'nq'; Z=RAW/'nq.zip'; URL='https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/nq.zip'
CORP=NQ/'corpus.jsonl'; QUER=NQ/'queries.jsonl'; QRELSF=NQ/'qrels'/'test.tsv'
if not (CORP.is_file() and QUER.is_file() and QRELSF.is_file()):
 with requests.get(URL,stream=True,timeout=180) as r:
  r.raise_for_status()
  with open(Z,'wb') as f:
   for c in r.iter_content(8*1024*1024):
    if c: f.write(c)
 with zipfile.ZipFile(Z) as z: z.extractall(RAW)
OFF=CACHE/'corpus_offsets.npy'; IDS=CACHE/'corpus_docids.txt'
if not OFF.is_file():
 offs=[]; ids=[]
 with open(CORP,'rb') as f:
  while True:
   p=f.tell(); line=f.readline()
   if not line: break
   o=json.loads(line); offs.append(p); ids.append(str(o['_id']))
 np.save(OFF,np.asarray(offs,dtype=np.int64)); IDS.write_text(chr(10).join(ids))
offs=np.load(OFF,mmap_mode='r'); ids=IDS.read_text().splitlines(); d2r={d:i for i,d in enumerate(ids)}
qtext={str(o['_id']):str(o.get('text','')) for o in map(json.loads,open(QUER,encoding='utf-8'))}
qdf=pd.read_csv(QRELSF,sep='\t'); qc=next(c for c in ['query-id','query_id','qid'] if c in qdf); dc=next(c for c in ['corpus-id','corpus_id','doc_id'] if c in qdf); sc=next((c for c in ['score','relevance','rel'] if c in qdf),None)
qdf[qc]=qdf[qc].astype(str); qdf[dc]=qdf[dc].astype(str)
if sc: qdf=qdf[pd.to_numeric(qdf[sc],errors='coerce')>0]
rels=defaultdict(set)
for _,r in qdf.iterrows():
 q,d=str(r[qc]),str(r[dc])
 if q in qtext and d in d2r: rels[q].add(d2r[d])
qids=sorted([q for q in rels if rels[q]],key=split_score); m=len(qids)//2; FIT=qids[:m]; VALID=qids[m:]
sm={'n_all':len(qids),'n_fit':len(FIT),'n_valid':len(VALID),'fit_sha':hashlib.sha256(chr(10).join(FIT).encode()).hexdigest(),'valid_sha':hashlib.sha256(chr(10).join(VALID).encode()).hexdigest()}; (RUN/'v035_query_split_manifest.json').write_text(json.dumps(sm,indent=2)); print(sm)


{'n_all': 3452, 'n_fit': 1726, 'n_valid': 1726, 'fit_sha': 'dadb30b47e4903f1261094aa39fd795cd8cc7bc0a22789f7d3241b21e2f0dbed', 'valid_sha': 'c68342c57c73a92f214b01af7fb62452e835efae3f9796fc9112c12c213f2442'}


In [5]:
assert torch.cuda.is_available(),'A100-class GPU recommended'
enc=SentenceTransformer(MODEL_ID,device='cuda'); actual=int(enc.get_sentence_embedding_dimension()); print(MODEL_ID,actual); assert actual==DIM
fh=open(CORP,'rb')
def doc(row): fh.seek(int(offs[int(row)])); return json.loads(fh.readline())
def text(o):
 t=str(o.get('title','') or '').strip(); b=str(o.get('text','') or '').strip(); return (t+' '+b).strip() if t else b


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/618 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

thenlper/gte-base 768


/tmp/ipykernel_3057/16434958.py:2: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  enc=SentenceTransformer(MODEL_ID,device='cuda'); actual=int(enc.get_sentence_embedding_dimension()); print(MODEL_ID,actual); assert actual==DIM


In [6]:
N=len(offs); nb=math.ceil(N/BLOCK)
for b in range(nb):
 s=b*BLOCK; e=min(N,(b+1)*BLOCK); p=EMB/f'block_{b:04d}_{s}_{e}.npy'
 if p.is_file():
  a=np.load(p,mmap_mode='r')
  if a.shape==(e-s,DIM) and a.dtype==np.float16: print('skip',b+1,'/',nb); continue
 xs=[text(doc(i)) for i in range(s,e)]; v=enc.encode(xs,batch_size=256,convert_to_numpy=True,normalize_embeddings=True,show_progress_bar=True).astype(np.float16); np.save(p,v); del xs,v; gc.collect(); torch.cuda.empty_cache()
print('embedding blocks ready')


Batches:   0%|          | 0/196 [00:00<?, ?it/s]

Batches:   0%|          | 0/196 [00:00<?, ?it/s]

Batches:   0%|          | 0/196 [00:00<?, ?it/s]

Batches:   0%|          | 0/196 [00:00<?, ?it/s]

Batches:   0%|          | 0/196 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/196 [00:00<?, ?it/s]

Batches:   0%|          | 0/196 [00:00<?, ?it/s]

Batches:   0%|          | 0/196 [00:00<?, ?it/s]

Batches:   0%|          | 0/123 [00:00<?, ?it/s]

embedding blocks ready


In [7]:
meta=[]
for p in sorted(EMB.glob('block_*.npy')):
 z=re.match(r'block_(\d+)_(\d+)_(\d+)\.npy',p.name)
 if z:
  b,s,e=map(int,z.groups()); meta.append((b,s,e,p))
arr={b:np.load(p,mmap_mode='r') for b,s,e,p in meta}
def gather(rows):
 rows=np.asarray(rows,dtype=np.int64); flat=rows.ravel(); out=np.empty((len(flat),DIM),np.float32); bids=flat//BLOCK
 for b in np.unique(bids):
  mask=bids==b; local=flat[mask]-int(b)*BLOCK; out[mask]=np.asarray(arr[int(b)][local],np.float32)
 return norm_rows(out).reshape(*rows.shape,DIM)
QEP=CACHE/'qemb.npy'
if not QEP.is_file(): np.save(QEP,enc.encode([qtext[q] for q in qids],batch_size=256,convert_to_numpy=True,normalize_embeddings=True).astype(np.float32))
qemb=np.load(QEP,mmap_mode='r'); qpos={q:i for i,q in enumerate(qids)}
TR=CACHE/'train_rows.npy'
if not TR.is_file(): np.save(TR,np.sort(np.random.default_rng(SEED).choice(N,min(TRAIN_DOCS,N),replace=False)).astype(np.int64))
train=gather(np.load(TR)); print('train',train.shape)


train (250000, 768)


In [8]:
PQP=IDX/'pq64.faiss'; SQP=IDX/'sq8.faiss'
def new_pq():
 q=faiss.IndexFlatIP(DIM); x=faiss.IndexIVFPQ(q,DIM,NLIST,PQ_M,PQ_NBITS,faiss.METRIC_INNER_PRODUCT); x.train(train); return x
def new_sq():
 q=faiss.IndexFlatIP(DIM); x=faiss.IndexIVFScalarQuantizer(q,DIM,NLIST,faiss.ScalarQuantizer.QT_8bit,faiss.METRIC_INNER_PRODUCT); x.train(train); return x
pq=faiss.read_index(str(PQP)) if PQP.is_file() else new_pq(); sq=faiss.read_index(str(SQP)) if SQP.is_file() else new_sq()
if pq.ntotal!=N or sq.ntotal!=N:
 pq,sq=new_pq(),new_sq()
 for b,s,e,p in tqdm(meta,desc='index add'):
  x=norm_rows(np.asarray(arr[b],np.float32)); pq.add(x); sq.add(x)
 faiss.write_index(pq,str(PQP)); faiss.write_index(sq,str(SQP))
print(pq.ntotal,sq.ntotal)


index add:   0%|          | 0/54 [00:00<?, ?it/s]

2681468 2681468


In [9]:
faiss.omp_set_num_threads(os.cpu_count() or 1); ps=faiss.ParameterSpace()
def search(idx,Q,npb): ps.set_index_parameter(idx,'nprobe',int(npb)); return idx.search(np.ascontiguousarray(Q,np.float32),TOPK)
def ndcg(rows,rel):
 dcg=sum(1/math.log2(i+2) for i,r in enumerate(rows[:UK]) if int(r) in rel); ideal=min(len(rel),UK); return 0. if ideal==0 else dcg/sum(1/math.log2(i+2) for i in range(ideal))
def one(idx,ids,npb):
 Q=np.stack([np.asarray(qemb[qpos[q]],np.float32) for q in ids]); D,I=search(idx,Q,npb); return np.asarray([ndcg(r,rels[q]) for q,r in zip(ids,I)]),I
hi,_=one(sq,FIT,HIGH_NPROBE); rp,_=one(pq,FIT,HIGH_NPROBE); rloss=float(np.mean(hi-rp)); rows=[]
for npb in NPROBE_GRID:
 lo,_=one(sq,FIT,npb); sl=float(np.mean(hi-lo)); mm=abs(sl-rloss); rows.append({'nprobe':npb,'rep_loss':rloss,'search_loss':sl,'abs_mismatch':mm,'relative_mismatch':mm/abs(rloss) if abs(rloss)>1e-12 else None})
cal=pd.DataFrame(rows).sort_values(['abs_mismatch','nprobe']); SELECT=int(cal.iloc[0].nprobe); cal.to_csv(RUN/'v035_fit_calibration.csv',index=False); gate={'selected_nprobe':SELECT,'fit_rep_loss':rloss,'selected_search_loss':float(cal.iloc[0].search_loss),'relative_mismatch':float(cal.iloc[0].relative_mismatch),'valid_inspected':False}; (RUN/'v035_calibration_gate.json').write_text(json.dumps(gate,indent=2)); CAL_SHA=sha_file(RUN/'v035_calibration_gate.json'); (RUN/'V035_CALIBRATION_SHA256.txt').write_text(CAL_SHA); print(cal, 'SELECT',SELECT)


   nprobe  rep_loss  search_loss  abs_mismatch  relative_mismatch
2       4  0.108455     0.099144      0.009311           0.085855
1       2  0.108455     0.149374      0.040919           0.377286
3       8  0.108455     0.058676      0.049779           0.458983
4      16  0.108455     0.026089      0.082366           0.759446
5      32  0.108455     0.011878      0.096577           0.890478
0       1  0.108455     0.219618      0.111162           1.024961 SELECT 4


In [10]:
(RUN/'VALID_STARTED.txt').write_text(datetime.now(timezone.utc).isoformat())
vh,_=one(sq,VALID,HIGH_NPROBE); vr,_=one(pq,VALID,HIGH_NPROBE); vs,_=one(sq,VALID,SELECT); oo=pd.DataFrame({'query_id':VALID,'high':vh,'rep_low':vr,'search_low':vs}); oo['rep_loss']=oo.high-oo.rep_low; oo['search_loss']=oo.high-oo.search_low; oo.to_csv(RUN/'v035_valid_oneshot.csv',index=False); print(oo.mean(numeric_only=True))


high           0.510578
rep_low        0.414236
search_low     0.422983
rep_loss       0.096342
search_loss    0.087595
dtype: float64


In [11]:
def feed(rows,scores,kind,k=20,tau=.1):
 v=gather(np.asarray(rows[:k],dtype=np.int64))
 if kind=='mean': f=v.mean(0)
 else:
  s=np.asarray(scores[:k],float); w=np.exp((s-s.max())/tau); w/=w.sum(); f=np.sum(v*w[:,None],axis=0)
 return f/max(float(np.linalg.norm(f)),1e-12)
def jd(a,b):
 A,B=set(map(int,a)),set(map(int,b)); return 1-len(A&B)/len(A|B)
q0=norm_rows(np.stack([np.asarray(qemb[qpos[q]],np.float32) for q in VALID])); rec=[]
for pi,pol in enumerate(POLICIES):
 st={'high':q0.copy(),'rep':q0.copy(),'search':q0.copy()}
 for t in range(H+1):
  ret={}; util={}
  for b in st:
   idx,npb=(sq,HIGH_NPROBE) if b=='high' else ((pq,HIGH_NPROBE) if b=='rep' else (sq,SELECT)); D,I=search(idx,st[b],npb); ret[b]=(D,I); util[b]=np.asarray([ndcg(r,rels[q]) for q,r in zip(VALID,I)])
  for i,q in enumerate(VALID):
   for mech,b in [('representation','rep'),('search_effort','search')]:
    sg=float(util['high'][i]-util[b][i]); rec.append({'query_id':q,'policy_id':pi,'round':t,'mechanism':mech,'semantic':1-float(np.dot(st['high'][i],st[b][i])),'candidate':jd(ret['high'][1][i],ret[b][1][i]),'signed':sg,'absolute':abs(sg)})
  pd.DataFrame(rec).to_parquet(RUN/'v035_trajectory_partial.parquet',index=False)
  if t==H: break
  for b in st:
   D,I=ret[b]; nxt=np.empty_like(st[b])
   for i in range(len(VALID)):
    f=feed(I[i],D[i],pol['kind'],20,pol['tau'] or .1); x=(1-pol['alpha'])*q0[i]+pol['alpha']*f; nxt[i]=x/max(float(np.linalg.norm(x)),1e-12)
   st[b]=nxt
traj=pd.DataFrame(rec); traj.to_parquet(RUN/'v035_trajectory.parquet',index=False); print('trajectory rows',len(traj))


trajectory rows 138080


In [12]:
ep=[]
for (q,pi,m),g in traj.groupby(['query_id','policy_id','mechanism']):
 g=g.sort_values('round'); ep.append({'query_id':q,'policy_id':pi,'mechanism':m,'H1':ols(g.semantic),'H2':ols(g.candidate),'H3_abs':ols(g.absolute),'H3_signed':ols(g.signed),'terminal_abs':float(g.absolute.iloc[-1]),'max_abs':float(g.absolute.max()),'mean_abs':float(g.absolute.mean())})
ep=pd.DataFrame(ep); ep.to_parquet(RUN/'v035_query_policy_endpoints.parquet',index=False); qe=ep.groupby(['query_id','mechanism'],as_index=False).mean(numeric_only=True); qe.to_csv(RUN/'v035_query_level_endpoints.csv',index=False)
metrics=['H1','H2','H3_abs','H3_signed','terminal_abs','max_abs','mean_abs']; w=qe.pivot(index='query_id',columns='mechanism',values=metrics)
def contrast(m,off): return boot_mean(w[(m,'representation')].to_numpy()-w[(m,'search_effort')].to_numpy(),SEED+off)
primary=contrast('H3_abs',1); lo,hi=primary['ci95']; cls='POSITIVE_TRANSFER' if lo>0 else ('REVERSED_TRANSFER' if hi<0 else 'UNRESOLVED'); result={'study_id':'ARC-v0.35','encoder':MODEL_ID,'dimension':DIM,'selected_nprobe':SELECT,'n_fit':len(FIT),'n_valid':len(VALID),'primary':{**primary,'classification':cls},'secondary':{m:contrast(m,10+i) for i,m in enumerate([x for x in metrics if x!='H3_abs'])},'prior_384':PRIOR_384,'protocol_sha':PROTOCOL_SHA,'calibration_sha':CAL_SHA,'retuned_after_valid':False}; (RUN/'v035_primary_gate.json').write_text(json.dumps(result,indent=2)); print(json.dumps(result,indent=2))


{
  "study_id": "ARC-v0.35",
  "encoder": "thenlper/gte-base",
  "dimension": 768,
  "selected_nprobe": 4,
  "n_fit": 1726,
  "n_valid": 1726,
  "primary": {
    "n": 1726,
    "mean": 0.013214982712503825,
    "ci95": [
      0.0109938052664159,
      0.015488102454562757
    ],
    "classification": "POSITIVE_TRANSFER"
  },
  "secondary": {
    "H1": {
      "n": 1726,
      "mean": 0.0002775915856295085,
      "ci95": [
        0.00021372916741320395,
        0.0003423160818639629
      ]
    },
    "H2": {
      "n": 1726,
      "mean": 0.011294753936437316,
      "ci95": [
        0.010270154651504966,
        0.012332881771352965
      ]
    },
    "H3_signed": {
      "n": 1726,
      "mean": 0.017553154582605694,
      "ci95": [
        0.015285279696974243,
        0.019850963912390863
      ]
    },
    "terminal_abs": {
      "n": 1726,
      "mean": 0.11252666942699042,
      "ci95": [
        0.10034412175737537,
        0.12459058434060215
      ]
    },
    "max_abs": {


In [13]:
p=result['primary']
if p['classification']=='POSITIVE_TRANSFER': wording=f'Frozen 768-d GTE-base replication preserved the short-horizon ordering: representation-minus-search H3_abs={p["mean"]:+.6f}, 95% CI [{p["ci95"][0]:+.6f},{p["ci95"][1]:+.6f}], after FIT selected nprobe={SELECT}. This transfers the finding to one 768-d encoder only.'
elif p['classification']=='REVERSED_TRANSFER': wording='Frozen 768-d replication reversed the prior ordering; report as a model-scale/geometry boundary.'
else: wording='Frozen 768-d replication was unresolved; report as scale-transfer uncertainty.'
(RUN/'v035_final_report.json').write_text(json.dumps({'result':result,'calibration':gate,'suggested_manuscript_wording':wording},indent=2)); print(wording)
rows=[]
for pth in sorted(RUN.iterdir()):
 if pth.is_file() and pth.name!='V035_ARTIFACT_SHA256.csv': rows.append({'file':pth.name,'bytes':pth.stat().st_size,'sha256':sha_file(pth)})
pd.DataFrame(rows).to_csv(RUN/'V035_ARTIFACT_SHA256.csv',index=False)


Frozen 768-d GTE-base replication preserved the short-horizon ordering: representation-minus-search H3_abs=+0.013215, 95% CI [+0.010994,+0.015488], after FIT selected nprobe=4. This transfers the finding to one 768-d encoder only.


## Reporting guardrail
A positive outcome supports transfer to **one** 768-d GTE-base setting. It does not establish dimension invariance, generality to every large encoder, late interaction, or cost-matched deployment equivalence.
